In [ ]:
import torch
import numpy as np

from src.knots import *
from src.extensions import *
from src.full_models import HyperbolicMinimalSurfacePINN
from src.plotting import plot_error, plot_knot, montecarlo_error, plot_mu_heatmap_log
from src.geometry import minimal_in_H4_PDE_flat_new
from src.samplers import MixSampler
from src.training import train_PINN_Adam, refine_PINN_lbfgs
from src.double_point_analysis import find_candidate_double_points, refine_double_points_newton
from src.interior_models import MLP

from mpl_toolkits.mplot3d import Axes3D
%matplotlib widget

## Load a pre-trained model

Pre-trained checkpoints live in `trained_models/`. Uncomment the file you want to inspect and re-run the cell.

In [ ]:
# load the model
trained_models_path = 'trained_models/paper/'
knot_type = 'stevedore' #'torus'
knot_parameters = 'R1.6_mirrorFalse' #'R1_p3_q2_r0.5'
perturbed = True

model_name = trained_models_path + 'KNOT_' + knot_type + '_KNOT_PAR_' + knot_parameters
if perturbed:
    model_name += '_PERTURBED.pt'
else:
    model_name += '.pt'

model = HyperbolicMinimalSurfacePINN.load(model_name)

## Plot the boundary knot

The curve $\gamma(\theta) \subset \mathbb{R}^3$ that the trained minimal surface bounds at $\partial D^2$.

In [ ]:
# show the knot
knot_evaluator = model.get_knot().get_evaluator()
plot_knot(knot_evaluator, dtype=model.kwargs['dtype'])

## Error statistics

Two views of the trained surface:

1. Mean squared $L^2$ error of the PDE residual and its pointwise max on a random sample.
2. A 2D heatmap of $\|t(x)\|^2$ over the disc — useful for spotting where the network is least accurate.

In [ ]:
# Same diagnostics on a freshly instantiated (UNTRAINED) model with the
# same architecture — shown here as a baseline before training.
untrained_model = HyperbolicMinimalSurfacePINN(**model.kwargs)

sampler = MixSampler(target=(0.2, 0.03), mix=0.7, bias=0.5, sigma=0.3, dtype=untrained_model.kwargs['dtype'])

xy = sampler(1000)
t = minimal_in_H4_PDE_flat_new(untrained_model)(xy)

print('Max pointwise norm of the tension field:', torch.max((t**2).sum(dim=-1)**0.5).detach().item())
print('MSE:', (t**2).sum(dim=-1).mean().detach().item())

In [ ]:
# plot the error on a regular grid
plot_error(
    minimal_in_H4_PDE_flat_new(untrained_model),
    dtype=untrained_model.kwargs['dtype'],
    vmin=0,
    vmax=1,
    grid_size=300,
    boundary_linewidth=2.6,
    figsize=(6,5),
    colorbar_label='',
    title=None)

In [ ]:
# compute the squared L2 error and the max of the pointwise norm of the tension field
sampler = MixSampler(target=(0.2, 0.03), mix=0.7, bias=0.5, sigma=0.3, dtype=model.kwargs['dtype'])

xy = sampler(1000)
t = minimal_in_H4_PDE_flat_new(model)(xy)

print('Max pointwise norm of the tension field:', torch.max((t**2).sum(dim=-1)**0.5).detach().item())
print('MSE:', (t**2).sum(dim=-1).mean().detach().item())

In [ ]:
plot_error(
    minimal_in_H4_PDE_flat_new(model),
    dtype=untrained_model.kwargs['dtype'],
    vmin=0,
    vmax=1,
    grid_size=300,
    boundary_linewidth=2.6,
    figsize=(6,5),
    colorbar_label='',
    title=None)

In [ ]:
# montecarlo_error(
#     minimal_in_H4_PDE_flat_new(model),
#     num_samples=1000,
#     size_samples=2**14,
#     figsize=(5,3),
#     bins=100,
#     title=None,
#     xlabel = 'Loss'
# )

## Self-proximity map

In [ ]:
_ = plot_mu_heatmap_log(
    model,
    epsilon=0.2,
    candidates=None,
    grid_resolution=200,
)

In [ ]:
candidates = find_candidate_double_points(
    model,
    grid_resolution=200,
    epsilon=0.2,
    tau=0.03,
    dtype=model.kwargs['dtype'])

print('Number of candidates:', len(candidates))

In [ ]:
refined_pairs, jacobians, final_distances = refine_double_points_newton(
    model,
    candidates,
    tol=1e-14,
    max_iter=200,
    dedup_tol=1e-11,
    dtype=model.kwargs['dtype'],
)

# Pre-calculate once to improve performance
arr_pairs = np.array(refined_pairs)
arr_jacobians = np.array(jacobians)

# Summaries
pos_mask = arr_jacobians > 0
neg_mask = arr_jacobians <= 0
print(f"Positive double points: {np.sum(pos_mask)}")
print(f"Negative double points: {np.sum(neg_mask)}")
print("-" * 60)

# Header (Updated width for the Dist column to accommodate scientific notation)
print(f"{'#':<5} | {'Type':<10} | {'Jacobian':<10} | {'Codomain Distance':<18} | {'p1 (x,y)':<20} | {'p2 (x,y)':<20}")
print("-" * 96)

for i, (pair, jac, dist) in enumerate(zip(refined_pairs, jacobians, final_distances)):
    pt_type = "Positive" if jac > 0 else "Negative"
    
    # Format coordinates
    p1 = f"({pair[0][0]:.3f}, {pair[0][1]:.3f})"
    p2 = f"({pair[1][0]:.3f}, {pair[1][1]:.3f})"
    
    # dist:<18.2e aligns with the 18-character width of the new header
    print(f"{i+1:<5} | {pt_type:<10} | {jac:<10.3f} | {dist:<18.2e} | {p1:<20} | {p2:<20}")

In [ ]:
# Bundle A - color
_ = plot_mu_heatmap_log(
    # --- Main settings
    u_callable = model,
    epsilon = 0.3,
    cmap='viridis_r',
    grid_resolution=300,
    figsize=(6,5),
    # title = None,

    # # --- Candidate Settings ---
    # candidates=candidates,
    # candidate_color="#E85D04",
    # alpha = 0.8,

    # --- Refined pairs Settings ---
    refined_pairs=refined_pairs,
    jacobians=jacobians,
    refined_pos_color='#ffc000',
    refined_neg_color="#fe0c0c",

    # --- Legend Settings ---
    # legend_title = 'Double points',
    legend_label_candidates='Candidate double points',
    legend_label_refined_pos=r'Positive double points',
    legend_label_refined_neg=r'Negative double points',
    # legend_bbox_to_anchor=None,
    # legend_loc='upper right',
    legend_frameon=True,

    # --- Boundary Settings ---
    boundary_color='black', 
    boundary_linewidth=2.6,

    # --- Colorbar Settings ---
    show_colorbar=True,
    colorbar_label=None,
)

In [ ]:
# Bundle B - greyscale
plot_mu_heatmap_log(
    # --- Main settings
    u_callable = model,
    epsilon = 0.3,
    cmap='gray',
    grid_resolution=300,
    figsize=(6,5),
    # title = None,

    # # --- Candidate Settings ---
    # candidates=candidates,
    # candidate_color="#444444",
    # alpha = 0.8,

    # --- Refined pairs Settings ---
    refined_pairs=refined_pairs,
    jacobians=jacobians,
    refined_pos_color="#000000",
    refined_neg_color="#888888",

    # --- Legend Settings ---
    # legend_title = 'Double points',
    legend_label_candidates='Candidate double points',
    legend_label_refined_pos=r'Positive double points',
    legend_label_refined_neg=r'Negative double points',
    # legend_bbox_to_anchor=None,
    # legend_loc='upper right',
    legend_frameon=True,

    # --- Boundary Settings ---
    boundary_color='black', 
    boundary_linewidth=2.6,

    # --- Colorbar Settings ---
    show_colorbar=True,
    colorbar_label=None,
)